# Bài 55: Phát hiện từ vựng mở (Open-Vocabulary Detection)

Bộ phát hiện ở Bài 41 dự đoán một trong một tập nhỏ, cố định các lớp được định sẵn lúc huấn luyện — chính xác là giới hạn phân loại ở Bài 36, chỉ là áp dụng cho hộp thay vì toàn bộ ảnh. **Phát hiện từ vựng mở (open-vocabulary detection)** (Grounding DINO, OWL-ViT) loại bỏ giới hạn đó bằng cách tách việc phát hiện thành hai kỹ năng tách biệt được: *tìm* đối tượng (một kỹ năng không phân biệt lớp, có khả năng chuyển giao đáng ngạc nhiên giữa các danh mục đối tượng) và *gọi tên* chúng (giao cho một phép so khớp vector nhúng văn bản kiểu CLIP, Bài 52, có thể nhận ra bất kỳ danh mục nào mô tả được bằng lời). Bài này xây dựng cả hai nửa và kiểm tra sự kết hợp trên một danh mục mà bộ hồi quy hộp của bộ phát hiện chưa bao giờ nhận được một hộp có nhãn nào.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## Lớp cơ sở và một lớp mới

Ba hình dạng: dấu cộng, hình tròn, hình vuông. Bộ hồi quy hộp sẽ chỉ luôn thấy các hộp có nhãn cho hai trong số chúng — các **lớp cơ sở (base classes)**. Hình vuông là **lớp mới (novel class)**: bộ phát hiện không bao giờ thấy một hình vuông nào được gán nhãn hộp trong khi huấn luyện, chỉ có dấu cộng và hình tròn.

In [ ]:
SIZE = 32
SHAPES = ['plus', 'circle', 'square']
BASE_SHAPES = ['plus', 'circle']
NOVEL_SHAPE = 'square'

def make_image(shape_type, cx, cy, size=SIZE, r=6):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-r:cx+r+1] = 1.0
        img[cy-r:cy+r+1, cx-1:cx+2] = 1.0
    elif shape_type == 'circle':
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= r**2] = 1.0
    else:  # square
        img[cy-r:cy+r+1, cx-r:cx+r+1] = 1.0
    return img

def make_detection_example(rng, shapes, size=SIZE, r=6):
    shape_type = rng.choice(shapes)
    cx, cy = rng.integers(r + 2, size - r - 2), rng.integers(r + 2, size - r - 2)
    img = make_image(shape_type, cx, cy)
    img = np.clip(img + rng.normal(0, 0.05, img.shape), 0, 1).astype(np.float32)
    box = np.array([cx - r, cy - r, 2 * r, 2 * r], dtype=np.float32) / size
    return img, box, shape_type

def crop_and_resize(img, box, out_size=SIZE):
    x0, y0, w, h = (box * SIZE).astype(int)
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(SIZE, x0 + max(w, 1)), min(SIZE, y0 + max(h, 1))
    crop = img[y0:y1, x0:x1]
    if crop.size == 0:
        return np.zeros((out_size, out_size), dtype=np.float32)
    resized = F.interpolate(torch.tensor(crop[None, None]), size=(out_size, out_size),
                             mode='bilinear', align_corners=False)
    return resized[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))
demo_rng = np.random.default_rng(42)
for ax, shape in zip(axes, SHAPES):
    im, box, _ = make_detection_example(demo_rng, [shape])
    ax.imshow(im, cmap='gray')
    x0, y0, w, h = box * SIZE
    ax.add_patch(patches.Rectangle((x0, y0), w, h, edgecolor='lime', facecolor='none', linewidth=2))
    ax.set_title(f'{shape}  (base={shape in BASE_SHAPES})', fontsize=9)
    ax.axis('off')
plt.show()

## Nửa 1: một bộ hồi quy hộp không phân biệt lớp

Hồi quy hộp ở Bài 41, giữ nguyên, ngoại trừ mục tiêu không bao giờ là một nhãn lớp — chỉ là "đối tượng ở đâu", được huấn luyện độc quyền trên các cảnh dấu cộng và hình tròn.

In [ ]:
class BoxRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1))
        self.fc = nn.Linear(32, 4)

    def forward(self, x):
        return torch.sigmoid(self.fc(self.conv(x).flatten(1)))

def iou_batch(pred, target):
    px0, py0, pw, ph = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]
    tx0, ty0, tw, th = target[:, 0], target[:, 1], target[:, 2], target[:, 3]
    px1, py1, tx1, ty1 = px0 + pw, py0 + ph, tx0 + tw, ty0 + th
    ix0, iy0 = torch.maximum(px0, tx0), torch.maximum(py0, ty0)
    ix1, iy1 = torch.minimum(px1, tx1), torch.minimum(py1, ty1)
    inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
    union = pw * ph + tw * th - inter
    return inter / union.clamp(min=1e-8)

rng = np.random.default_rng(1)
N = 400
Xtr, Btr = [], []
for _ in range(N):
    img, box, _ = make_detection_example(rng, BASE_SHAPES)  # base classes only
    Xtr.append(img); Btr.append(box)
Xtr, Btr = np.array(Xtr, dtype=np.float32), np.array(Btr, dtype=np.float32)

torch.manual_seed(0)
box_model = BoxRegressor()
opt = torch.optim.Adam(box_model.parameters(), lr=0.005)
Xt, Bt = torch.tensor(Xtr).unsqueeze(1), torch.tensor(Btr)
for _ in range(400):
    opt.zero_grad()
    loss = F.mse_loss(box_model(Xt), Bt)
    loss.backward()
    opt.step()

# test localization on ALL THREE shapes, including the never-boxed novel square
test_rng = np.random.default_rng(2)
Xte, Bte, labels_te = [], [], []
for _ in range(150):
    img, box, lbl = make_detection_example(test_rng, SHAPES)
    Xte.append(img); Bte.append(box); labels_te.append(lbl)
Xte, Bte = np.array(Xte, dtype=np.float32), np.array(Bte, dtype=np.float32)

with torch.no_grad():
    pred_boxes = box_model(torch.tensor(Xte).unsqueeze(1))
ious = iou_batch(pred_boxes, torch.tensor(Bte)).numpy()

for shape in SHAPES:
    mask = np.array([l == shape for l in labels_te])
    print(f'{shape:>8} (base={shape in BASE_SHAPES}): mean box IoU = {ious[mask].mean():.3f}')

Khả năng định vị chuyển giao sang hình dạng mới khá tốt — chưa bao giờ thấy một hộp hình vuông có nhãn, bộ hồi quy vẫn tìm ra hình vuông với IoU đáng nể, dù kém chính xác thấy rõ so với các lớp cơ sở mà nó thực sự được huấn luyện. "Có đối tượng ở đây không" không phân biệt lớp là một kỹ năng chung chung hơn, dễ chuyển giao hơn so với danh tính lớp chính xác.

## Nửa 2: một bộ phân loại kiểu CLIP, được huấn luyện trên cả ba lớp

Đây là thứ đại diện cho "tiền huấn luyện rộng" — một bộ mã hóa ảnh/văn bản kép (Bài 52), được huấn luyện tương phản trên các vùng cắt có nhãn của **cả ba** hình dạng, dấu cộng, hình tròn và hình vuông. Các bộ phát hiện từ vựng mở thực tế dựa vào chính sự bất đối xứng này: bộ mã hóa ảnh-văn bản được tiền huấn luyện trên một tập từ vựng rộng hơn rất nhiều (hàng trăm triệu cặp ảnh-chú thích) so với những gì chú thích hộp của bất kỳ tập dữ liệu phát hiện nào từng bao phủ, nên nó đã "biết" một hình vuông trông như thế nào dù dữ liệu huấn luyện hộp của bộ phát hiện chưa bao giờ có hình vuông nào.

In [ ]:
VOCAB = ['a', 'photo', 'of', 'plus', 'circle', 'square']
W2ID = {w: i for i, w in enumerate(VOCAB)}

def caption_for(shape):
    return [W2ID[w] for w in ['a', 'photo', 'of', shape]]

class ImageEncoder(nn.Module):
    def __init__(self, out_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1))
        self.proj = nn.Linear(32, out_dim)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, out_dim=16, embed_dim=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.proj = nn.Linear(embed_dim, out_dim)

    def forward(self, ids):
        return self.proj(self.embed(ids).mean(dim=1))

def clip_loss(img_emb, txt_emb, temperature=0.1):
    img_emb = F.normalize(img_emb, dim=1)
    txt_emb = F.normalize(txt_emb, dim=1)
    logits = img_emb @ txt_emb.T / temperature
    targets = torch.arange(img_emb.shape[0])
    return (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2

# train on CROPS, matching how the detector will feed it patches at inference time --
# training on full uncropped images instead would create a train/test mismatch in object scale
clip_rng = np.random.default_rng(3)
N_CLIP = 400
X_clip, cap_clip = [], []
for _ in range(N_CLIP):
    shape = clip_rng.choice(SHAPES)
    cx, cy = clip_rng.integers(8, SIZE - 8), clip_rng.integers(8, SIZE - 8)
    img = make_image(shape, cx, cy, r=6)
    img = np.clip(img + clip_rng.normal(0, 0.05, img.shape), 0, 1).astype(np.float32)
    box = np.array([cx - 6, cy - 6, 12, 12], dtype=np.float32) / SIZE
    X_clip.append(crop_and_resize(img, box)); cap_clip.append(caption_for(shape))
X_clip = np.array(X_clip, dtype=np.float32)
cap_clip = np.array(cap_clip, dtype=np.int64)

torch.manual_seed(1)
img_enc = ImageEncoder()
txt_enc = TextEncoder(len(VOCAB))
opt2 = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()), lr=0.01)
caps_tensor = torch.tensor(cap_clip)
n_clip = len(X_clip)
for epoch in range(300):
    idx = np.random.default_rng(epoch).permutation(n_clip)[:64]
    loss = clip_loss(img_enc(torch.tensor(X_clip[idx]).unsqueeze(1)), txt_enc(caps_tensor[idx]))
    opt2.zero_grad()
    loss.backward()
    opt2.step()

print(f'CLIP training final loss: {loss.item():.3f}')

## Quy trình đầy đủ: phát hiện, cắt, phân loại bằng so khớp văn bản

Với mỗi ảnh kiểm tra: chạy bộ hồi quy hộp không phân biệt lớp, cắt vùng dự đoán, nhúng nó, và so sánh với các vector nhúng lời nhắc văn bản cho cả ba tên hình dạng — kể cả "square", thứ mà bộ hồi quy hộp chưa bao giờ được huấn luyện. Để cô lập *lý do* xảy ra bất kỳ lỗi nào, hãy đánh giá thêm cùng bộ phân loại đó dùng hộp *chuẩn* thay vì hộp dự đoán, loại bỏ hoàn toàn sai số định vị khỏi bức tranh.

In [ ]:
with torch.no_grad():
    prompt_embs = {s: F.normalize(txt_enc(torch.tensor([caption_for(s)])), dim=1) for s in SHAPES}

def classify_crop(img, box):
    with torch.no_grad():
        crop = crop_and_resize(img, box)
        emb = F.normalize(img_enc(torch.tensor(crop[None, None])), dim=1)
        sims = {s: (emb @ prompt_embs[s].T).item() for s in SHAPES}
    return max(sims, key=sims.get)

eval_rng = np.random.default_rng(2)
n_correct_pred, n_correct_gt, n_total = {s: 0 for s in SHAPES}, {s: 0 for s in SHAPES}, {s: 0 for s in SHAPES}
for _ in range(150):
    img, gt_box, true_shape = make_detection_example(eval_rng, SHAPES)
    with torch.no_grad():
        pred_box = box_model(torch.tensor(img[None, None])).squeeze(0).numpy()
    pred_using_predicted_box = classify_crop(img, pred_box)
    pred_using_gt_box = classify_crop(img, gt_box)
    n_total[true_shape] += 1
    n_correct_pred[true_shape] += (pred_using_predicted_box == true_shape)
    n_correct_gt[true_shape] += (pred_using_gt_box == true_shape)

print(f'{"class":>8} {"base?":>7} {"end-to-end (predicted box)":>28} {"classification only (true box)":>32}')
for s in SHAPES:
    acc_pred = n_correct_pred[s] / n_total[s]
    acc_gt = n_correct_gt[s] / n_total[s]
    print(f'{s:>8} {str(s in BASE_SHAPES):>7} {acc_pred:>27.1%} {acc_gt:>31.1%}')

Với hộp chuẩn, phân loại đạt 100% trên cả ba lớp — bộ phân loại kiểu CLIP thực sự nhận ra "square" chỉ bằng lời nhắc văn bản, dù *bộ phát hiện* chưa một lần được huấn luyện trên hộp của một hình vuông. Điều đó xác nhận gọn gàng tuyên bố cốt lõi của từ vựng mở. Nhưng ở chế độ đầu-cuối, dùng hộp dự đoán của chính bộ phát hiện, độ chính xác với hình vuông tụt lại xa phía sau các lớp cơ sở — không phải vì bộ phân loại quên mất hình vuông trông như thế nào, mà vì các hộp của bộ hồi quy hộp không phân biệt lớp cho hình dạng mới kém chính xác một cách đo được (nhớ lại khoảng cách IoU ở Nửa 1), và một vùng cắt bị méo khó phân loại đúng hơn ngay cả với một bộ phân loại hoàn toàn có năng lực. *Phân loại* danh mục mới và *định vị* danh mục mới là hai bài toán thực sự riêng biệt với các kiểu thất bại riêng biệt, và thí nghiệm này cô lập chính xác cái nào chịu trách nhiệm cho mỗi lỗi đầu-cuối cụ thể — chính xác là phép chẩn đoán mà các bài báo phát hiện từ vựng mở thực tế báo cáo (AP lớp cơ sở và AP lớp mới, được tách riêng) khi mô tả hệ thống của họ vẫn còn thiếu sót ở đâu.

### Bài tập

1. Tăng kích thước tập huấn luyện cho `box_model` từ 400 lên 2000 mẫu lớp cơ sở. Nhiều dữ liệu huấn luyện lớp cơ sở hơn có cải thiện IoU định vị lớp mới (hình vuông) không, dù bản thân hình vuông vẫn không bao giờ có trong nhãn huấn luyện?
2. Thêm một hình dạng thứ tư (ví dụ một tam giác nhỏ, theo mẫu ở Bài 36) là mới đối với *cả* bộ hồi quy hộp lẫn bộ phân loại CLIP. Quy trình thất bại ở giai đoạn định vị, giai đoạn phân loại, hay cả hai — và bạn sẽ phân biệt chúng thế nào bằng phép chẩn đoán hộp chuẩn của bài này?
3. `crop_and_resize` luôn đổi kích thước về cùng một kích thước đầu ra bất kể kích thước của hộp dự đoán. Các bộ phát hiện từ vựng mở thực tế nhạy cảm với sai số kích thước hộp chính vì lý do này — một hộp có *tỉ lệ khung hình* sai làm méo vùng cắt trước khi bộ phân loại kịp thấy nó. Hãy đo tương quan giữa IoU của hộp dự đoán và tính đúng đắn của phân loại trên các trường hợp kiểm tra hình vuông: những hình vuông được định vị tệ nhất có cũng thường là những hình bị phân loại sai không?